In [6]:
import numpy as np
import string
from collections import defaultdict
from sklearn.model_selection import train_test_split

# Larger dataset (Spam = 1, Not Spam = 0)
X = [
    "Win a free iPhone now", "Congratulations! You won a lottery", "Hurry up! Limited-time discount",
    "Exclusive deal for you", "Reminder: Your subscription is expiring", "Project deadline tomorrow",
    "Meeting at 4 PM", "Let's catch up for coffee", "Urgent: Update your account details",
    "Claim your free gift now", "Final call: 50% off on your order", "Call me when you’re free",
    "Your invoice is attached", "Job opportunity for you", "Important security update",
    "Earn money fast with this trick", "Order confirmed. Expected delivery: tomorrow",
    "Can you send the report today?", "Reminder: Doctor’s appointment at 5 PM", "Claim your prize now!",
    "Don't miss out on this offer!", "Lunch at 1 PM?", "Best investment tips for you",
    "Your Amazon package has been shipped", "Bank notice: Verify your account to avoid suspension",
    "Scam alert: Do not share your OTP", "Earn $5000 from home", "Weekend sale up to 70% off",
    "How was your weekend?", "Meeting rescheduled to 2 PM"
]

y = [1, 1, 1, 1, 0, 0, 0, 0, 1,
     1, 1, 0, 0, 0, 0, 1, 0, 0,
     0, 1, 1, 0, 1, 0, 1, 1, 1,
     1, 0, 0]  # 1 = Spam, 0 = Not Spam

# Text preprocessing: Lowercase, remove punctuation
def clean_text(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Naïve Bayes classifier from scratch
class NaiveBayes:
    def __init__(self):
        self.class_probs = {}  # P(Y)
        self.word_probs = defaultdict(lambda: defaultdict(float))  # P(X|Y)

    def fit(self, X, y):
        class_counts = defaultdict(int)
        word_counts = defaultdict(lambda: defaultdict(int))
        total_words = defaultdict(int)

        # Count words for each class
        for text, label in zip(X, y):
            class_counts[label] += 1
            words = clean_text(text)
            for word in words:
                word_counts[label][word] += 1
                total_words[label] += 1

        # Compute P(Y) (prior probabilities)
        total_samples = len(y)
        self.class_probs = {label: class_counts[label] / total_samples for label in class_counts}

        # Compute P(X|Y) using Laplace smoothing
        vocab = set(word for label in word_counts for word in word_counts[label])
        vocab_size = len(vocab)

        for label in class_counts:
            self.word_probs[label] = {
                word: (word_counts[label][word] + 1) / (total_words[label] + vocab_size)
                for word in vocab
            }

    def predict(self, X):
        predictions = []
        for text in X:
            words = clean_text(text)
            scores = {}

            for label in self.class_probs:
                log_prob = np.log(self.class_probs[label])  # Log(P(Y))
                for word in words:
                    log_prob += np.log(self.word_probs[label].get(word, 1e-6))  # Log(P(X|Y))
                scores[label] = log_prob

            predictions.append(max(scores, key=scores.get))  # Pick the label with max probability

        return predictions

# Train and evaluate the model
nb = NaiveBayes()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

# Compute accuracy
accuracy = sum(1 for true, pred in zip(y_test, y_pred) if true == pred) / len(y_test)
print(f"Accuracy: {accuracy:.2f}")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Best alpha: 0.1
Accuracy: 0.9370748299319728

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.94       292
           1       0.93      0.95      0.94       296

    accuracy                           0.94       588
   macro avg       0.94      0.94      0.94       588
weighted avg       0.94      0.94      0.94       588

